# Proyecto 8 - Visualización estadística con Seaborn

**Actividad 2.1 - Data Science Showcase**  
**Equipo:** B  
**Proyecto:** 8 - Visualización estadística con Seaborn  
**Tema:** Heatmaps y distribuciones  
**Dataset:** Gapminder World completo (1952-2007)

---

## Objetivo

Analizar y visualizar el comportamiento de diferentes variables del dataset Gapminder mediante la biblioteca **Seaborn**.

En este proyecto se utilizarán **gráficas de distribución** y **mapas de calor (Heatmaps)** para identificar la forma en que se distribuyen los datos y observar relaciones entre variables numéricas.

El propósito es demostrar cómo la visualización estadística facilita la interpretación de los datos y permite encontrar patrones que pueden ser difíciles de identificar únicamente mediante tablas.


## 1. Importación de librerías

Primero se importan las librerías necesarias para trabajar con los datos y generar las visualizaciones.

- **Pandas:** manipulación y análisis de datos.
- **Seaborn:** creación de visualizaciones estadísticas.
- **Matplotlib:** configuración y presentación de las gráficas.
- **Pathlib:** creación de la carpeta donde se guardarán las figuras.


In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Crear carpeta para guardar las figuras
Path('figuras').mkdir(exist_ok=True)

# Estilo general de Seaborn
sns.set_theme(style='whitegrid')

print('Librerías cargadas correctamente.')


Librerías cargadas correctamente.


## 2. Carga del dataset

Se utilizará el dataset **Gapminder**, que contiene información de diferentes países y años.

Entre sus variables se encuentran:

- `country`: país.
- `continent`: continente.
- `year`: año.
- `life_exp`: esperanza de vida.
- `population`: población.
- `gdp_cap`: PIB per cápita.

El código intenta primero encontrar `gapminder_full.csv` en los archivos disponibles en Colab. Si no se encuentra, utiliza una copia pública del mismo dataset.


In [2]:
from pathlib import Path
import pandas as pd

possible_files = list(Path('.').rglob('gapminder_full.csv'))

if possible_files:
    csv_path = possible_files[0]
    print(f'Archivo encontrado: {csv_path}')
    df = pd.read_csv(csv_path)
else:
    # Respaldo: fuente pública del dataset Gapminder
    url = 'https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv'
    df = pd.read_csv(url)

    # Homologar nombres para trabajar con la estructura usada en el proyecto
    df = df.rename(columns={
        'lifeExp': 'life_exp',
        'pop': 'population',
        'gdpPercap': 'gdp_cap'
    })

df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

print('Dataset cargado correctamente.')
print(f'Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas')
df.head()


Archivo encontrado: gapminder_full.csv
Dataset cargado correctamente.
Dimensiones: 1704 filas y 6 columnas


## 3. Exploración inicial

Antes de realizar las visualizaciones, es importante conocer la estructura del dataset.

Con `info()` podemos observar los tipos de datos y con `describe()` obtenemos estadísticas descriptivas de las variables numéricas.


In [3]:
print('Información general del dataset:')
df.info()

print('\nEstadísticas descriptivas:')
display(df.describe())


Información general del dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   country     1704 non-null   object 
 1   year        1704 non-null   int64  
 2   population  1704 non-null   int64  
 3   continent   1704 non-null   object 
 4   life_exp    1704 non-null   float64
 5   gdp_cap     1704 non-null   float64
dtypes: float64(2), int64(2), object(2)
memory usage: 80.0+ KB

Estadísticas descriptivas:
             year    population     life_exp        gdp_cap
count  1704.00000  1.704000e+03  1704.000000    1704.000000
mean   1979.50000  2.960121e+07    59.474439    7215.327081
std      17.26533  1.061579e+08    12.917107    9857.454543
min    1952.00000  6.001100e+04    23.599000     241.165876
25%    1965.75000  2.793664e+06    48.198000    1202.060309
50%    1979.50000  7.023596e+06    60.712500    3531.846989
75%    1993.25000  1.9

## 4. Revisión de valores nulos

La calidad de los datos es importante antes de generar análisis estadísticos.

Se cuentan los valores nulos de cada columna para comprobar si existen datos faltantes.


In [4]:
nulos = df.isna().sum().sort_values(ascending=False)

print('Valores nulos por columna:')
display(nulos.to_frame('valores_nulos'))


Valores nulos por columna:
            valores_nulos
country                 0
year                    0
population              0
continent               0
life_exp                0
gdp_cap                 0


## 5. Distribución de la esperanza de vida

Una **distribución** permite observar cómo se encuentran repartidos los valores de una variable.

En este caso se utilizará `sns.histplot()` para representar la esperanza de vida. También se agregará una curva **KDE (Kernel Density Estimate)** para observar la forma general de la distribución.


In [5]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=df,
    x='life_exp',
    kde=True
)

plt.title('Distribución de la esperanza de vida')
plt.xlabel('Esperanza de vida')
plt.ylabel('Frecuencia')

plt.tight_layout()
plt.savefig(
    'figuras/01_distribucion_esperanza_vida.png',
    dpi=160,
    bbox_inches='tight'
)

plt.show()


### Interpretación

El histograma permite identificar en qué rangos se concentra la esperanza de vida de los registros del dataset.

La curva KDE ayuda a visualizar la forma de la distribución de manera continua. Esto permite observar si los valores están concentrados alrededor de determinados rangos o si existe mayor dispersión.


## 6. Distribución de esperanza de vida por continente

Seaborn permite incorporar una variable categórica mediante `hue`.

En esta visualización se comparará la distribución de la esperanza de vida entre los diferentes continentes.


In [6]:
plt.figure(figsize=(11, 7))

sns.histplot(
    data=df,
    x='life_exp',
    hue='continent',
    kde=True,
    element='step',
    stat='density',
    common_norm=False
)

plt.title('Distribución de la esperanza de vida por continente')
plt.xlabel('Esperanza de vida')
plt.ylabel('Densidad')

plt.tight_layout()
plt.savefig(
    'figuras/02_distribucion_por_continente.png',
    dpi=160,
    bbox_inches='tight'
)

plt.show()


### Interpretación

Esta gráfica permite comparar visualmente la forma de las distribuciones entre continentes.

El uso de `hue='continent'` permite diferenciar cada grupo y observar que los rangos y concentraciones de esperanza de vida no son iguales para todos los continentes.


## 7. Distribución del PIB per cápita

También podemos estudiar cómo se distribuye una variable económica.

Debido a que el PIB per cápita presenta una escala amplia, se utilizará una escala logarítmica en el eje X para facilitar la visualización.


In [7]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=df,
    x='gdp_cap',
    kde=True
)

plt.xscale('log')
plt.title('Distribución del PIB per cápita')
plt.xlabel('PIB per cápita (escala logarítmica)')
plt.ylabel('Frecuencia')

plt.tight_layout()
plt.savefig(
    'figuras/03_distribucion_pib_per_capita.png',
    dpi=160,
    bbox_inches='tight'
)

plt.show()


## 8. Matriz de correlación

Para construir un **Heatmap**, primero se calcula una matriz de correlación.

La correlación permite medir la relación lineal entre variables numéricas:

- Un valor cercano a **1** indica una relación positiva fuerte.
- Un valor cercano a **-1** indica una relación negativa fuerte.
- Un valor cercano a **0** indica una relación lineal débil.


In [8]:
variables_numericas = [
    'year',
    'life_exp',
    'population',
    'gdp_cap'
]

correlacion = df[variables_numericas].corr()

print('Matriz de correlación:')
display(correlacion)


Matriz de correlación:
                year  life_exp  population   gdp_cap
year        1.000000  0.435611    0.082308  0.227318
life_exp    0.435611  1.000000    0.064955  0.583706
population  0.082308  0.064955    1.000000 -0.025600
gdp_cap     0.227318  0.583706   -0.025600  1.000000


## 9. Heatmap de correlaciones

El **Heatmap** representa los valores de la matriz de correlación mediante colores.

La función `sns.heatmap()` facilita la identificación visual de relaciones entre variables. `annot=True` muestra el valor numérico de cada correlación dentro de las celdas.


In [9]:
plt.figure(figsize=(9, 7))

sns.heatmap(
    correlacion,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5,
    vmin=-1,
    vmax=1
)

plt.title('Heatmap de correlación de variables numéricas')

plt.tight_layout()
plt.savefig(
    'figuras/04_heatmap_correlacion.png',
    dpi=160,
    bbox_inches='tight'
)

plt.show()


### Interpretación del Heatmap

El mapa de calor permite identificar rápidamente qué variables presentan relaciones más fuertes.

La intensidad y dirección de la relación se observan mediante la escala de colores y los valores numéricos mostrados en cada celda.

Es importante recordar que **correlación no significa causalidad**: una correlación elevada no demuestra por sí sola que una variable sea la causa de otra.


## 10. Heatmap de correlación por continente

Como análisis adicional, podemos calcular la correlación utilizando únicamente los datos de un continente.

En este ejemplo se selecciona **América**, si está disponible en el dataset.


In [10]:
continente_objetivo = 'Americas'

if continente_objetivo in df['continent'].unique():
    df_america = df[df['continent'] == continente_objetivo]

    corr_america = df_america[variables_numericas].corr()

    plt.figure(figsize=(9, 7))

    sns.heatmap(
        corr_america,
        annot=True,
        cmap='coolwarm',
        fmt='.2f',
        linewidths=0.5,
        vmin=-1,
        vmax=1
    )

    plt.title('Heatmap de correlación - América')

    plt.tight_layout()
    plt.savefig(
        'figuras/05_heatmap_america.png',
        dpi=160,
        bbox_inches='tight'
    )

    plt.show()
else:
    print('El continente seleccionado no está disponible en el dataset.')


## 11. ¿Qué aporta Seaborn al análisis de datos?

Seaborn permite transformar información numérica en representaciones visuales que facilitan el análisis.

En este proyecto se utilizaron principalmente:

| Herramienta | Función |
|---|---|
| `sns.histplot()` | Analizar distribuciones |
| `kde=True` | Mostrar una estimación de densidad |
| `hue` | Diferenciar grupos |
| `sns.heatmap()` | Representar matrices mediante colores |
| `.corr()` | Calcular correlaciones |

Estas herramientas forman parte del **Análisis Exploratorio de Datos (EDA)**.


## 12. Conclusiones

A través de este proyecto se comprobó que Seaborn es una herramienta útil para realizar visualizaciones estadísticas en Python.

Las gráficas de distribución permitieron observar cómo se encuentran repartidos los valores de variables como la esperanza de vida y el PIB per cápita.

Por otra parte, los Heatmaps permitieron representar visualmente las correlaciones entre diferentes variables numéricas, haciendo más sencillo identificar relaciones positivas o negativas.

La combinación de **Pandas, Seaborn y Matplotlib** permite realizar un análisis exploratorio más completo, ya que no solamente se observan los datos en forma de tabla, sino que también se pueden identificar patrones mediante representaciones gráficas.

Finalmente, es importante considerar que una correlación representa una relación estadística y no necesariamente una relación de causa y efecto.


## 13. Respuesta breve para la exposición

**¿Qué es Seaborn?**  
Es una biblioteca de Python especializada en visualización estadística y construida sobre Matplotlib.

**¿Qué es un Heatmap?**  
Es una representación gráfica en la que los valores de una matriz se muestran mediante colores. Es útil para visualizar correlaciones.

**¿Qué es una distribución?**  
Es la manera en que los valores de una variable se encuentran repartidos dentro de un conjunto de datos.

**¿Para qué sirve `kde=True`?**  
Agrega una curva de estimación de densidad que ayuda a visualizar la forma de la distribución.

**¿Para qué sirve `.corr()`?**  
Calcula la correlación entre variables numéricas.

**¿Por qué es importante este proyecto?**  
Porque antes de aplicar técnicas de Machine Learning es necesario comprender los datos. Las visualizaciones ayudan a encontrar patrones, relaciones y características importantes del conjunto de datos.
